# Vanilla FGSM

Companion notebook for *FGSM Variants in White Box Settings: A Comparative Study*, using ImageNet-pretrained ResNet50V2.

Implemented from the paper’s one-step formula (section III-C); the standalone original Vanilla notebook was not recovered.

[Paper PDF](https://doi.org/10.1109/ICICT60155.2024.10544606) · [Reported Tables I and II](../docs/FINAL_PAPER_RESULTS.md)


## Setup

Run in a Colab TensorFlow runtime, or install TensorFlow, NumPy, Matplotlib, and Pillow in your local Python environment. The first model load downloads the ImageNet weights.


In [ ]:
import json
import platform
from pathlib import Path
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
model = tf.keras.applications.ResNet50V2(include_top=True, weights='imagenet')
model.trainable = False


## Input image

Set `IMAGE_PATH` to a local photo. In Colab, leaving the default missing path opens an upload picker. See [source notes](../docs/SOURCES.md) for input-image availability.

`LABEL_INDEX = 283` is the ImageNet Persian-cat label for the paper's illustrated input. For a different photo, set its true ImageNet class index. Setting it to `None` uses the model's initial prediction as a surrogate label.


In [ ]:
IMAGE_PATH = 'cat.jpg'
LABEL_INDEX = 283
EPSILONS = [0.025, 0.05, 0.075, 0.1]  # Paper Table I

if not Path(IMAGE_PATH).is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Set IMAGE_PATH to an existing image file.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one input image.')
    IMAGE_PATH = next(iter(uploaded))

photo = tf.keras.utils.load_img(IMAGE_PATH, target_size=(224, 224))
image = tf.keras.utils.img_to_array(photo)
image = tf.keras.applications.resnet_v2.preprocess_input(image)
image = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
initial = model(image, training=False)
label_index = int(tf.argmax(initial[0])) if LABEL_INDEX is None else int(LABEL_INDEX)
if not 0 <= label_index < int(initial.shape[-1]):
    raise ValueError('LABEL_INDEX must be between 0 and 999.')
print('Attack label index:', label_index)


## One-step attack

For each epsilon, start from the same clean image and compute `x_adv = clip(x + epsilon * sign(gradient), -1, 1)`. The attack uses categorical cross-entropy with the fixed label selected above.


In [ ]:
def fgsm(model, image, label_index, epsilon):
    """Return a one-step untargeted FGSM perturbation of a preprocessed image."""
    predictions = model(image, training=False)
    label = tf.one_hot(
        tf.reshape(tf.cast(label_index, tf.int32), [-1]),
        depth=tf.shape(predictions)[-1],
    )
    with tf.GradientTape() as tape:
        tape.watch(image)
        pred = model(image, training=False)
        loss = tf.keras.losses.categorical_crossentropy(label, pred)
    signed_gradient = tf.sign(tape.gradient(loss, image))
    return tf.clip_by_value(image + epsilon * signed_gradient, -1.0, 1.0)


## Visualize and record this run

The noise column rescales the signed perturbation for visibility. Confidence is the top-1 softmax score. These outputs are computed from the current input and environment; they do not replace Table I.


In [ ]:
def top1(tensor):
    scores = model(tensor, training=False).numpy()
    _, name, confidence = tf.keras.applications.resnet_v2.decode_predictions(scores, top=1)[0][0]
    return name, float(confidence) * 100

clean_name, clean_confidence = top1(image)
fig, axes = plt.subplots(len(EPSILONS), 3, figsize=(10, 3 * len(EPSILONS)), squeeze=False)
rows = []
for row, epsilon in enumerate(EPSILONS):
    adversarial = fgsm(model, image, label_index, epsilon)
    name, confidence = top1(adversarial)
    delta = adversarial - image
    noise_display = delta[0].numpy() / (2 * epsilon) + 0.5
    axes[row, 0].imshow(np.clip(image[0].numpy() * 0.5 + 0.5, 0, 1))
    axes[row, 0].set_title(f'Input: {clean_name}\n{clean_confidence:.4f}%')
    axes[row, 1].imshow(np.clip(noise_display, 0, 1))
    axes[row, 1].set_title(f'Noise (rescaled), epsilon={epsilon}')
    axes[row, 2].imshow(np.clip(adversarial[0].numpy() * 0.5 + 0.5, 0, 1))
    axes[row, 2].set_title(f'{name}\n{confidence:.4f}%')
    for ax in axes[row]:
        ax.axis('off')
    rows.append({'epsilon': epsilon, 'top1_class': name, 'confidence_pct': confidence})
plt.tight_layout()
plt.show()
print(json.dumps(rows, indent=2))


## Paper reference

Table I (page 4) reports the following Vanilla FGSM measurements. This is a transcription for reference, not notebook output.

| Epsilon | Class | Confidence |
| ---: | --- | ---: |
| 0.025 | Egyptian cat | 38.5972% |
| 0.05 | Egyptian cat | 39.2992% |
| 0.075 | Tiger cat | 40.0466% |
| 0.1 | Tiger cat | 47.7527% |

[Source notes](../docs/SOURCES.md) document the available research material and printed parameter differences.
